# Coastal ecosystem connectivity: mangrove patch proximity and avoided EAD linkage

This notebook identifies which mangrove patches are in proximity to coral reefs and/or seagrass at **250 m**, **500 m**, and **1000 m**. It then links those proximity results to the current **weighted area-distance mangrove attribution** outputs for coastal-flood avoided EADs.

The notebook produces:
- a patch-level table for all mangrove patches with nearest coral and seagrass distances
- proximity flags and proximity classes at 250 m, 500 m, and 1000 m
- merged minimum- and maximum-scenario net avoided EAD summaries for each mangrove patch
- summary tables showing the extent to which **positive net avoided EAD** patches are near coral reefs, seagrass, both, or neither

The analysis uses the current Jamaica mangrove patch layer and the current coral and seagrass layers held in the project data folders.


In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.float_format = "{:,.2f}".format

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
mangrove_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"
coral_path = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/corals/Coral Reefs.shp"
seagrass_path = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/seagrass/Seagrass.shp"
minimum_patch_ead_csv = (
    base_path
    / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.csv"
)
maximum_patch_ead_csv = (
    base_path
    / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates/mangrove_attribution_area_distance_all_sectors_signed/mangrove_attribution_total_all_sectors_signed_area_distance_5000m_nn_fallback.csv"
)
output_dir = base_path / "dphil_paper_3/results/co_benefits/connectivity/coastal_connectivity"
output_dir.mkdir(parents=True, exist_ok=True)

jamaica_metric_grid_crs = "EPSG:3448"
proximity_thresholds_m = [250, 500, 1000]
proximity_class_order = ["both", "coral_only", "seagrass_only", "neither"]


## Inputs

- Mangrove patches are taken from the same Forces of Nature mangrove patch layer used in the coastal-flood attribution notebooks.
- Coral reefs and seagrass are taken from the current common incoming land-cover layers.
- Avoided EAD linkage uses the current **signed weighted area-distance attribution** patch summaries for the minimum and maximum scenarios.

Distances are calculated as the **minimum edge-to-edge distance** from each mangrove patch polygon to the coral reef layer and to the seagrass layer. Threshold flags are then assigned from those minimum distances.


In [ ]:
def load_geodataframe(input_path, target_crs):
    geodataframe = gpd.read_file(input_path)
    if geodataframe.crs is None:
        raise ValueError(f"CRS is missing for {input_path}")
    if str(geodataframe.crs).upper() != target_crs:
        geodataframe = geodataframe.to_crs(target_crs)
    return geodataframe


def load_mangrove_patches(mangrove_path, target_crs):
    mangrove_patches = load_geodataframe(mangrove_path, target_crs)
    if "ID" in mangrove_patches.columns:
        mangrove_patches["mangrove_patch_id"] = mangrove_patches["ID"].astype(int)
    else:
        mangrove_patches = mangrove_patches.reset_index(drop=True)
        mangrove_patches["mangrove_patch_id"] = np.arange(1, len(mangrove_patches) + 1)

    mangrove_patches["area_ha"] = mangrove_patches.geometry.area / 10_000.0
    mangrove_patches["parish"] = mangrove_patches["Parish"] if "Parish" in mangrove_patches.columns else "unknown"
    mangrove_patches["mangrove_type"] = mangrove_patches["TYPE"] if "TYPE" in mangrove_patches.columns else "unknown"

    return mangrove_patches[["mangrove_patch_id", "area_ha", "parish", "mangrove_type", "geometry"]].copy()


def add_nearest_distance_column(source_geodataframe, target_geodataframe, distance_column):
    if target_geodataframe.empty:
        raise ValueError(f"Target geodataframe for {distance_column} is empty.")
    target_union_geometry = target_geodataframe.geometry.union_all()
    output_geodataframe = source_geodataframe.copy()
    output_geodataframe[distance_column] = output_geodataframe.geometry.distance(target_union_geometry)
    return output_geodataframe


def add_proximity_columns(patch_table, thresholds_m):
    output_table = patch_table.copy()
    for threshold_m in thresholds_m:
        coral_column = f"coral_within_{threshold_m}m"
        seagrass_column = f"seagrass_within_{threshold_m}m"
        both_column = f"both_within_{threshold_m}m"
        either_column = f"either_within_{threshold_m}m"
        proximity_class_column = f"proximity_class_{threshold_m}m"

        output_table[coral_column] = output_table["nearest_coral_distance_m"] <= threshold_m
        output_table[seagrass_column] = output_table["nearest_seagrass_distance_m"] <= threshold_m
        output_table[both_column] = output_table[coral_column] & output_table[seagrass_column]
        output_table[either_column] = output_table[coral_column] | output_table[seagrass_column]
        output_table[proximity_class_column] = np.select(
            [
                output_table[both_column],
                output_table[coral_column] & ~output_table[seagrass_column],
                ~output_table[coral_column] & output_table[seagrass_column],
            ],
            ["both", "coral_only", "seagrass_only"],
            default="neither",
        )
    return output_table


def load_patch_ead_table(all_patch_ids, patch_ead_csv, scenario_name):
    patch_ead_table = pd.read_csv(patch_ead_csv)[[
        "Mangrove_ID",
        "Net_Avoided_EAD_USD_attributed",
        "Positive_Avoided_EAD_USD_attributed",
        "Negative_Avoided_EAD_USD_attributed",
        "Gross_Avoided_EAD_USD_attributed",
        "Asset_Count",
        "Net_Impact_Class",
    ]].rename(
        columns={
            "Mangrove_ID": "mangrove_patch_id",
            "Net_Avoided_EAD_USD_attributed": f"net_avoided_ead_usd_{scenario_name}",
            "Positive_Avoided_EAD_USD_attributed": f"positive_avoided_ead_usd_{scenario_name}",
            "Negative_Avoided_EAD_USD_attributed": f"negative_avoided_ead_usd_{scenario_name}",
            "Gross_Avoided_EAD_USD_attributed": f"gross_avoided_ead_usd_{scenario_name}",
            "Asset_Count": f"asset_count_{scenario_name}",
            "Net_Impact_Class": f"net_impact_class_{scenario_name}",
        }
    )

    merged_table = all_patch_ids.merge(patch_ead_table, on="mangrove_patch_id", how="left")
    numeric_columns = [
        f"net_avoided_ead_usd_{scenario_name}",
        f"positive_avoided_ead_usd_{scenario_name}",
        f"negative_avoided_ead_usd_{scenario_name}",
        f"gross_avoided_ead_usd_{scenario_name}",
        f"asset_count_{scenario_name}",
    ]
    for column_name in numeric_columns:
        merged_table[column_name] = pd.to_numeric(merged_table[column_name], errors="coerce").fillna(0.0)
    merged_table[f"asset_count_{scenario_name}"] = merged_table[f"asset_count_{scenario_name}"].astype(int)
    merged_table[f"net_impact_class_{scenario_name}"] = merged_table[
        f"net_impact_class_{scenario_name}"
    ].fillna("no_attributed_ead")
    return merged_table


def build_patch_count_summary(patch_table, thresholds_m):
    summary_rows = []
    total_patch_count = len(patch_table)
    for threshold_m in thresholds_m:
        summary_rows.append(
            {
                "distance_m": threshold_m,
                "total_patch_count": total_patch_count,
                "patches_near_coral": int(patch_table[f"coral_within_{threshold_m}m"].sum()),
                "patches_near_seagrass": int(patch_table[f"seagrass_within_{threshold_m}m"].sum()),
                "patches_near_both": int(patch_table[f"both_within_{threshold_m}m"].sum()),
                "patches_near_either": int(patch_table[f"either_within_{threshold_m}m"].sum()),
                "patches_near_neither": int((patch_table[f"proximity_class_{threshold_m}m"] == "neither").sum()),
            }
        )
    return pd.DataFrame(summary_rows)


def build_positive_net_summary_by_class(patch_table, thresholds_m, scenario_name):
    net_avoided_ead_column = f"net_avoided_ead_usd_{scenario_name}"
    positive_patch_table = patch_table.loc[patch_table[net_avoided_ead_column] > 0].copy()
    total_positive_patch_count = len(positive_patch_table)
    total_positive_net_ead_usd = float(positive_patch_table[net_avoided_ead_column].sum())

    summary_rows = []
    for threshold_m in thresholds_m:
        proximity_class_column = f"proximity_class_{threshold_m}m"
        for proximity_class in proximity_class_order:
            class_patch_table = positive_patch_table.loc[
                positive_patch_table[proximity_class_column] == proximity_class
            ].copy()
            class_patch_count = len(class_patch_table)
            class_positive_net_ead_usd = float(class_patch_table[net_avoided_ead_column].sum())
            summary_rows.append(
                {
                    "scenario": scenario_name,
                    "distance_m": threshold_m,
                    "proximity_class": proximity_class,
                    "positive_net_patch_count": class_patch_count,
                    "share_of_positive_net_patches_pct": (
                        100.0 * class_patch_count / total_positive_patch_count if total_positive_patch_count > 0 else np.nan
                    ),
                    "positive_net_avoided_ead_usd": class_positive_net_ead_usd,
                    "share_of_positive_net_avoided_ead_pct": (
                        100.0 * class_positive_net_ead_usd / total_positive_net_ead_usd if total_positive_net_ead_usd > 0 else np.nan
                    ),
                }
            )
    return pd.DataFrame(summary_rows)


def build_positive_net_summary_collapsed(patch_table, thresholds_m, scenario_name):
    net_avoided_ead_column = f"net_avoided_ead_usd_{scenario_name}"
    positive_patch_table = patch_table.loc[patch_table[net_avoided_ead_column] > 0].copy()
    total_positive_patch_count = len(positive_patch_table)
    total_positive_net_ead_usd = float(positive_patch_table[net_avoided_ead_column].sum())

    summary_rows = []
    for threshold_m in thresholds_m:
        coral_column = f"coral_within_{threshold_m}m"
        seagrass_column = f"seagrass_within_{threshold_m}m"
        both_column = f"both_within_{threshold_m}m"
        either_column = f"either_within_{threshold_m}m"
        summary_rows.append(
            {
                "scenario": scenario_name,
                "distance_m": threshold_m,
                "positive_net_patch_count_total": total_positive_patch_count,
                "positive_net_patch_count_near_coral": int(positive_patch_table[coral_column].sum()),
                "positive_net_patch_count_near_seagrass": int(positive_patch_table[seagrass_column].sum()),
                "positive_net_patch_count_near_both": int(positive_patch_table[both_column].sum()),
                "positive_net_patch_count_near_either": int(positive_patch_table[either_column].sum()),
                "positive_net_patch_count_near_neither": int((positive_patch_table[f"proximity_class_{threshold_m}m"] == "neither").sum()),
                "share_positive_net_patch_count_near_coral_pct": (
                    100.0 * positive_patch_table[coral_column].sum() / total_positive_patch_count if total_positive_patch_count > 0 else np.nan
                ),
                "share_positive_net_patch_count_near_seagrass_pct": (
                    100.0 * positive_patch_table[seagrass_column].sum() / total_positive_patch_count if total_positive_patch_count > 0 else np.nan
                ),
                "share_positive_net_patch_count_near_both_pct": (
                    100.0 * positive_patch_table[both_column].sum() / total_positive_patch_count if total_positive_patch_count > 0 else np.nan
                ),
                "positive_net_avoided_ead_usd_total": total_positive_net_ead_usd,
                "positive_net_avoided_ead_usd_near_coral": float(positive_patch_table.loc[positive_patch_table[coral_column], net_avoided_ead_column].sum()),
                "positive_net_avoided_ead_usd_near_seagrass": float(positive_patch_table.loc[positive_patch_table[seagrass_column], net_avoided_ead_column].sum()),
                "positive_net_avoided_ead_usd_near_both": float(positive_patch_table.loc[positive_patch_table[both_column], net_avoided_ead_column].sum()),
                "positive_net_avoided_ead_usd_near_either": float(positive_patch_table.loc[positive_patch_table[either_column], net_avoided_ead_column].sum()),
                "positive_net_avoided_ead_usd_near_neither": float(positive_patch_table.loc[positive_patch_table[f"proximity_class_{threshold_m}m"] == "neither", net_avoided_ead_column].sum()),
            }
        )
    collapsed_summary = pd.DataFrame(summary_rows)
    for category_name in ["coral", "seagrass", "both", "either", "neither"]:
        collapsed_summary[f"share_positive_net_avoided_ead_{category_name}_pct"] = np.where(
            collapsed_summary["positive_net_avoided_ead_usd_total"] > 0,
            100.0 * collapsed_summary[f"positive_net_avoided_ead_usd_near_{category_name}"] / collapsed_summary["positive_net_avoided_ead_usd_total"],
            np.nan,
        )
    return collapsed_summary


In [ ]:
mangrove_patches = load_mangrove_patches(mangrove_path, jamaica_metric_grid_crs)
corals = load_geodataframe(coral_path, jamaica_metric_grid_crs)
seagrass = load_geodataframe(seagrass_path, jamaica_metric_grid_crs)

mangrove_patches = add_nearest_distance_column(
    mangrove_patches,
    corals,
    "nearest_coral_distance_m",
)
mangrove_patches = add_nearest_distance_column(
    mangrove_patches,
    seagrass,
    "nearest_seagrass_distance_m",
)
mangrove_patches = add_proximity_columns(mangrove_patches, proximity_thresholds_m)

all_patch_ids = mangrove_patches[["mangrove_patch_id"]].copy()
minimum_patch_ead = load_patch_ead_table(all_patch_ids, minimum_patch_ead_csv, "minimum")
maximum_patch_ead = load_patch_ead_table(all_patch_ids, maximum_patch_ead_csv, "maximum")

patch_proximity_ead = (
    mangrove_patches
    .merge(minimum_patch_ead, on="mangrove_patch_id", how="left")
    .merge(maximum_patch_ead, on="mangrove_patch_id", how="left")
    .sort_values("mangrove_patch_id")
    .reset_index(drop=True)
)

print(f"Mangrove patches loaded: {len(mangrove_patches)}")
print(f"Coral polygons loaded: {len(corals)}")
print(f"Seagrass polygons loaded: {len(seagrass)}")
display(patch_proximity_ead.drop(columns=["geometry"]).head())


In [ ]:
patch_count_summary = build_patch_count_summary(patch_proximity_ead, proximity_thresholds_m)
positive_net_summary_by_class = pd.concat(
    [
        build_positive_net_summary_by_class(patch_proximity_ead, proximity_thresholds_m, scenario_name)
        for scenario_name in ["minimum", "maximum"]
    ],
    ignore_index=True,
)
positive_net_summary_collapsed = pd.concat(
    [
        build_positive_net_summary_collapsed(patch_proximity_ead, proximity_thresholds_m, scenario_name)
        for scenario_name in ["minimum", "maximum"]
    ],
    ignore_index=True,
)

patch_proximity_ead_attributes = patch_proximity_ead.drop(columns=["geometry"]).copy()

patch_output_csv = output_dir / "mangrove_patch_proximity_to_coral_seagrass_and_ead_linkage.csv"
patch_output_gpkg = output_dir / "mangrove_patch_proximity_to_coral_seagrass_and_ead_linkage.gpkg"
patch_count_summary_csv = output_dir / "mangrove_patch_proximity_counts_by_distance.csv"
positive_net_summary_by_class_csv = output_dir / "mangrove_positive_net_ead_proximity_summary_by_class.csv"
positive_net_summary_collapsed_csv = output_dir / "mangrove_positive_net_ead_proximity_summary_collapsed.csv"

patch_proximity_ead_attributes.to_csv(patch_output_csv, index=False)
patch_proximity_ead.to_file(patch_output_gpkg, driver="GPKG")
patch_count_summary.to_csv(patch_count_summary_csv, index=False)
positive_net_summary_by_class.to_csv(positive_net_summary_by_class_csv, index=False)
positive_net_summary_collapsed.to_csv(positive_net_summary_collapsed_csv, index=False)

print("Saved outputs:")
for output_path in [
    patch_output_csv,
    patch_output_gpkg,
    patch_count_summary_csv,
    positive_net_summary_by_class_csv,
    positive_net_summary_collapsed_csv,
]:
    print(f" - {output_path}")

print("Patch counts by distance:")
display(patch_count_summary)

print("Positive net avoided EAD proximity summary (collapsed):")
display(positive_net_summary_collapsed)


In [ ]:
review_columns = [
    "mangrove_patch_id",
    "parish",
    "area_ha",
    "nearest_coral_distance_m",
    "nearest_seagrass_distance_m",
    "proximity_class_250m",
    "proximity_class_500m",
    "proximity_class_1000m",
    "net_avoided_ead_usd_minimum",
    "net_impact_class_minimum",
    "net_avoided_ead_usd_maximum",
    "net_impact_class_maximum",
]

print("Top 20 mangrove patches by minimum-scenario net avoided EAD:")
display(
    patch_proximity_ead[review_columns]
    .sort_values(["net_avoided_ead_usd_minimum", "mangrove_patch_id"], ascending=[False, True])
    .head(20)
)

print("Top 20 mangrove patches by maximum-scenario net avoided EAD:")
display(
    patch_proximity_ead[review_columns]
    .sort_values(["net_avoided_ead_usd_maximum", "mangrove_patch_id"], ascending=[False, True])
    .head(20)
)

print("Positive net avoided EAD proximity summary by class:")
display(positive_net_summary_by_class)
